### This notebook can be used to get a guidance to suggest `max_atoms` in `config.yaml` file

In [8]:
from pathlib import Path
from collections import Counter

import numpy as np
from rdkit import Chem


def generateMaxAtomsConfig(targetSmiles: str, increasePercent: float = 0.5) -> dict:
    """
    Generate max_atoms config from a target SMILES.
    Increases each atom count by the specified percentage (default 50%).
    Returns a dictionary with atom limits.
    """
    mol = Chem.MolFromSmiles(targetSmiles)
    if mol is None:
        raise ValueError(f"Could not parse SMILES: {targetSmiles}")

    atomCounter = Counter(atom.GetSymbol() for atom in mol.GetAtoms())

    atomsOfInterest = ['C', 'N', 'O', 'S']

    maxAtoms = {}
    print(f"\nTarget molecule SMILES: {targetSmiles}")
    print(f"\nAtom counts in target molecule:")
    for atom in atomsOfInterest:
        count = atomCounter.get(atom, 0)
        suggested = int(np.ceil(count * (1 + increasePercent)))
        suggested = max(suggested, 1)
        maxAtoms[atom] = suggested
        print(f"  {atom}: {count}")

    print(f"\nGenerated max_atoms config ({int(increasePercent * 100)}% increase):")
    print(f"max_atoms:")
    for atom in atomsOfInterest:
        names = {'C': 'Carbon', 'N': 'Nitrogen', 'O': 'Oxygen', 'S': 'Sulfur'}
        print(f"  {atom}: {maxAtoms[atom]}   # {names[atom]}")

    return maxAtoms


if __name__ == "__main__":
    # ---- User supplies only this ----
    targetSmiles = "CCC=C1OC(=O)[C@@H](C)[C@H]1O"

    maxAtomsConfig = generateMaxAtomsConfig(targetSmiles)


Target molecule SMILES: CCC=C1OC(=O)[C@@H](C)[C@H]1O

Atom counts in target molecule:
  C: 8
  N: 0
  O: 3
  S: 0

Generated max_atoms config (50% increase):
max_atoms:
  C: 12   # Carbon
  N: 1   # Nitrogen
  O: 5   # Oxygen
  S: 1   # Sulfur


### For a broader search space take `targetSmiles` from a data frame

In [7]:
from rdkit import Chem
from rdkit.Chem import rdMolDescriptors
from collections import Counter
import pandas as pd
import numpy as np

# Read the CSV file
startersDF = pd.read_csv("generated_molecules_diverseStarterSecondExtensionSMILES.csv")
print(f"Loaded {len(startersDF)} molecules")

# Count atoms for each molecule
atomCounts = []

for smi in startersDF['Canonical_SMILES']:
    mol = Chem.MolFromSmiles(smi)
    if mol is None:
        atomCounts.append({'C': 0, 'N': 0, 'O': 0, 'S': 0})
        continue
    
    atomCounter = Counter(atom.GetSymbol() for atom in mol.GetAtoms())
    atomCounts.append({
        'C': atomCounter.get('C', 0),
        'N': atomCounter.get('N', 0),
        'O': atomCounter.get('O', 0),
        'S': atomCounter.get('S', 0)
    })

atomCountsDf = pd.DataFrame(atomCounts)
startersDF = pd.concat([startersDF, atomCountsDf], axis=1)

# Print atom count ranges
print(f"\nAtom count ranges across {len(startersDF)} molecules:")
print(f"  C (Carbon):   min = {startersDF['C'].min()}, max = {startersDF['C'].max()}")
print(f"  N (Nitrogen): min = {startersDF['N'].min()}, max = {startersDF['N'].max()}")
print(f"  O (Oxygen):   min = {startersDF['O'].min()}, max = {startersDF['O'].max()}")
print(f"  S (Sulfur):   min = {startersDF['S'].min()}, max = {startersDF['S'].max()}")

# Print suggested max_atoms config (max values + 50% increase)
maxC = startersDF['C'].max()
maxN = startersDF['N'].max()
maxO = startersDF['O'].max()
maxS = startersDF['S'].max()

print(f"\nSuggested max_atoms config (max values + 50% increase for expanded search space):")
print(f"  C: {int(np.ceil(maxC * 1.5))}")
print(f"  N: {int(np.ceil(maxN * 1.5))}")
print(f"  O: {int(np.ceil(maxO * 1.5))}")
print(f"  S: {int(np.ceil(maxS * 1.5))}")

startersDF

Loaded 2054 molecules

Atom count ranges across 2054 molecules:
  C (Carbon):   min = 6, max = 21
  N (Nitrogen): min = 0, max = 4
  O (Oxygen):   min = 2, max = 6
  S (Sulfur):   min = 0, max = 1

Suggested max_atoms config (max values + 50% increase for expanded search space):
  C: 32
  N: 6
  O: 9
  S: 2


,Canonical_SMILES,starter_mod,firstExtension_module,secondExtension_mod,release_mod,Molecular_Weight,has_target_substructure,matching_patterns,C,N,O,S
0,CCC=C1OC(=O)[C@@H](C)[C@H]1O,"[""AT{'substrate': 'Methylmalonyl-CoA'}"", 'load...",AT(hmal) + KR(B1) + DH,"[""AT{'substrate': 'Methylmalonyl-CoA'}"", ""KR{'...","TE(cyclic, ring=2)",156.18,True,['pattern_2: O1C(=O)CCC1'],8,0,3,0
1,CCC=C1OC(=O)[C@H](O)C1=O,"[""AT{'substrate': 'Methylmalonyl-CoA'}"", 'load...",AT(hmal) + KR(B1) + DH,"[""AT{'substrate': 'hmal'}"", ""KR{'type': 'C1'}""...","TE(cyclic, ring=1)",156.14,True,['pattern_2: O1C(=O)CCC1'],7,0,4,0
2,CCC=C1OC(=O)CC1=O,"[""AT{'substrate': 'Methylmalonyl-CoA'}"", 'load...",AT(hmal) + KR(B1) + DH,"[""AT{'substrate': 'Malonyl-CoA'}"", 'loading: F...","TE(cyclic, ring=0)",140.14,True,['pattern_2: O1C(=O)CCC1'],7,0,3,0
3,CCC=C1C[C@H](CC)C(=O)O1,"[""AT{'substrate': 'Methylmalonyl-CoA'}"", 'load...",AT(hmal) + KR(B1) + DH,"[""AT{'substrate': 'emal'}"", ""KR{'type': 'B1'}""...","TE(cyclic, ring=0)",154.21,True,['pattern_2: O1C(=O)CCC1'],9,0,2,0
4,CCC=C1OC(=O)[C@H](C(C)C)[C@H]1O,"[""AT{'substrate': 'Methylmalonyl-CoA'}"", 'load...",AT(hmal) + KR(B1) + DH,"[""AT{'substrate': 'isobutmal'}"", ""KR{'type': '...","TE(cyclic, ring=2)",184.23,True,['pattern_2: O1C(=O)CCC1'],10,0,3,0
...,...,...,...,...,...,...,...,...,...,...,...,...
2049,C[C@H](C=C1OC(=O)[C@@H](C)[C@@H]1O)CNC(=O)[C@H...,"[""AT{'substrate': '3measp'}"", 'loading: True']",AT(hmal) + KR(B1) + DH,"[""AT{'substrate': 'Methylmalonyl-CoA'}"", ""KR{'...","TE(cyclic, ring=6)",256.30,True,['pattern_2: O1C(=O)CCC1'],12,2,4,0
2050,CCCCCC[C@H]1C(=O)OC(=C[C@@H](C)CNC(=O)[C@H](C)...,"[""AT{'substrate': '3measp'}"", 'loading: True']",AT(hmal) + KR(B1) + DH,"[""AT{'substrate': 'hexmal'}"", ""KR{'type': 'A1'...","TE(cyclic, ring=6)",326.44,True,['pattern_2: O1C(=O)CCC1'],17,2,4,0
2051,CCCC[C@H]1CC(=C[C@@H](C)CNC(=O)[C@H](C)N)OC1=O,"[""AT{'substrate': '3measp'}"", 'loading: True']",AT(hmal) + KR(B1) + DH,"[""AT{'substrate': 'butmal'}"", ""KR{'type': 'B1'...","TE(cyclic, ring=4)",282.38,True,['pattern_2: O1C(=O)CCC1'],15,2,3,0
2052,CC[C@H]1C(=O)OC(=C[C@@H](C)CNC(=O)[C@H](C)N)[C...,"[""AT{'substrate': '3measp'}"", 'loading: True']",AT(hmal) + KR(B1) + DH,"[""AT{'substrate': 'emal'}"", ""KR{'type': 'A1'}""...","TE(cyclic, ring=6)",270.33,True,['pattern_2: O1C(=O)CCC1'],13,2,4,0
